# Human Action Classifier - Pretrained Model
**Schuldt et al. 2004. Recognizing Human Actions: A Local SVM Approach**

In [5]:
import torch
import torch.nn.functional as F
import numpy as np
import cv2
import re
import matplotlib.pyplot as plt

from pathlib import Path
from collections import Counter
from sklearn.cluster import MiniBatchKMeans
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from joblib import load

In [6]:
# device = "mps" if torch.backends.mps.is_available() else "cpu"
device = "cpu"
print(device)

cpu


### Load Presaved Features

In [7]:
print("Loading presaved features...")
features = torch.load("artifact/kth_features.pt")
X_train = features["X_train"]
y_train = features["y_train"]
X_val = features["X_val"]
y_val = features["y_val"]
X_test = features["X_test"]
y_test = features["y_test"]

print("X_train shape:", X_train.shape, "y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape, "y_val shape:", y_val.shape)
print("X_test shape:", X_test.shape, "y_test shape:", y_test.shape)

Loading presaved features...
X_train shape: torch.Size([407, 400]) y_train shape: torch.Size([407])
X_val shape: torch.Size([96, 400]) y_val shape: torch.Size([96])
X_test shape: torch.Size([96, 400]) y_test shape: torch.Size([96])


### Train Classifier

In [8]:
def train_svm(X, y):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X.numpy())

    clf = LinearSVC(C=1.0, max_iter=5000)
    clf.fit(Xs, y.numpy())

    return clf, scaler

In [9]:
print("Training classifier...")
clf, scaler = train_svm(X_train, y_train)

Training classifier...


### Model Evaluation

In [11]:
def evaluate(clf, scaler, X, y):
    Xs = scaler.transform(X.numpy())
    y_pred = clf.predict(Xs)

    acc = accuracy_score(y.numpy(), y_pred)
    cm = confusion_matrix(y.numpy(), y_pred)

    return acc, cm

In [7]:
print("---- Validation ----")
val_acc, _ = evaluate(clf, scaler, X_val, y_val)
print(f"Val accuracy: {val_acc*100:.2f}%")

---- Validation ----
Val accuracy: 79.17%


In [8]:
print("---- Test ----")
test_acc, cm = evaluate(clf, scaler, X_test, y_test)
print(f"Test accuracy: {test_acc*100:.2f}%")
print(cm)

---- Test ----
Test accuracy: 71.88%
[[ 7  6  3  0  0  0]
 [ 1 14  0  1  0  0]
 [ 2  0 12  0  1  1]
 [ 0  0  1 13  2  0]
 [ 0  0  0  6  9  1]
 [ 0  1  0  1  0 14]]


In [1]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
def train_rf(X, y):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X.numpy())

    clf = RandomForestClassifier(n_estimators=500, random_state=0, n_jobs=-1)
    clf.fit(Xs, y.numpy())

    return clf, scaler

In [ ]:
print("Training Random Forest Classifier...")
clf_rf, scaler_rf = train_rf(X_train, y_train)

print("\n---- Validation ----")
val_acc, _ = evaluate(clf_rf, scaler_rf, X_val, y_val)
print(f"Val accuracy (RF): {val_acc*100:.2f}%")

print("\n---- Test ----")
test_acc, cm = evaluate(clf_rf, scaler_rf, X_test, y_test)
print(f"Test accuracy (RF): {test_acc*100:.2f}%")
print("Confusion Matrix:")
print(cm)

Training Random Forest Classifier...

---- Validation ----
Val accuracy (RF): 76.04%

---- Test ----
Test accuracy (RF): 70.83%
Confusion Matrix:
[[ 6 10  0  0  0  0]
 [ 2 13  1  0  0  0]
 [ 1  2 12  0  0  1]
 [ 0  0  0  9  3  4]
 [ 0  0  0  2 12  2]
 [ 0  0  0  0  0 16]]


In [ ]:
from sklearn.neural_network import MLPClassifier

def train_mlp(X, y):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X.numpy())

    # hidden_layer_sizes: (512, 256)
    clf = MLPClassifier(hidden_layer_sizes=(512, 256), max_iter=1000, random_state=0)
    clf.fit(Xs, y.numpy())

    return clf, scaler

print("Training MLP Classifier...")
clf_mlp, scaler_mlp = train_mlp(X_train, y_train)

Training MLP Classifier...


In [16]:
# ---- Validation ----
print("---- Validation ----")
val_acc, _ = evaluate(clf_mlp, scaler_mlp, X_val, y_val)
print(f"Val accuracy (MLP): {val_acc*100:.2f}%")

# ---- Test ----
print("\n---- Test ----")
test_acc, cm = evaluate(clf_mlp, scaler_mlp, X_test, y_test)
print(f"Test accuracy (MLP): {test_acc*100:.2f}%")
print("Confusion Matrix:")
print(cm)

---- Validation ----
Val accuracy (MLP): 82.29%

---- Test ----
Test accuracy (MLP): 75.00%
Confusion Matrix:
[[ 6  5  4  0  0  1]
 [ 1 12  2  0  0  1]
 [ 2  0 13  0  0  1]
 [ 0  0  0 12  3  1]
 [ 0  0  0  1 13  2]
 [ 0  0  0  0  0 16]]


In [17]:
import xgboost as xgb

In [ ]:
def train_xgboost(X, y):
    X_np = X.numpy()
    y_np = y.numpy()

    clf = xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=0,
        use_label_encoder=False,
        eval_metric='mlogloss'
    )
    
    clf.fit(X_np, y_np)

    return clf

In [21]:
print("Training XGBoost Classifier...")
clf_xgb = train_xgboost(X_train, y_train)

# ---- Validation ----
print("\n---- Validation ----")
y_val_pred = clf_xgb.predict(X_val.numpy())
val_acc = accuracy_score(y_val.numpy(), y_val_pred)
print(f"Val accuracy (XGB): {val_acc*100:.2f}%")

# ---- Test ----
print("\n---- Test ----")
y_test_pred = clf_xgb.predict(X_test.numpy())
test_acc = accuracy_score(y_test.numpy(), y_test_pred)
test_cm = confusion_matrix(y_test.numpy(), y_test_pred)

print(f"Test accuracy (XGB): {test_acc*100:.2f}%")
print("Confusion Matrix:")
print(test_cm)

Training XGBoost Classifier...


c:\Users\aabab\kth\kth_env\Lib\site-packages\xgboost\training.py:200: UserWarning: [17:42:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



---- Validation ----
Val accuracy (XGB): 71.88%

---- Test ----
Test accuracy (XGB): 73.96%
Confusion Matrix:
[[13  3  0  0  0  0]
 [ 2 12  1  0  0  1]
 [ 2  0 13  1  0  0]
 [ 0  0  0  9  3  4]
 [ 0  0  0  5  9  2]
 [ 0  0  0  1  0 15]]
